# Heart Disease UCI — Modeling & Experiment Tracking

**Assignment:** MLOps Assignment 01 (AIMLCZG523)
**Tasks 2 & 3:** Feature Engineering + Model Development, and Experiment Tracking

This notebook documents the modelling decisions. The production training logic lives
in `src/train.py` (importable, testable, CI-friendly); here we narrate the *why* and
show the comparison results.


In [1]:
import sys, os, json
sys.path.append(os.path.abspath('../src'))
import pandas as pd
from preprocess import load_clean, build_preprocessor, ALL_FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES, BINARY_FEATURES
pd.set_option('display.max_columns', None)

## 1. Feature engineering strategy

Driven by the EDA findings:

| Feature group | Features | Transformation | Rationale |
|---|---|---|---|
| Numeric | age, trestbps, chol, thalach, oldpeak | median impute → standard scale | scaling helps LogReg; median is robust to outliers |
| Categorical | cp, restecg, slope, thal, ca | mode impute → one-hot | multi-level, non-ordinal; disease rate varies sharply across levels |
| Binary | sex, fbs, exang | passthrough | already 0/1 |

All transformations live in a single `ColumnTransformer` so they are fit on
training folds only and re-applied identically at inference — no train/serve skew.


In [2]:
X, y, report = load_clean('../data/heart_disease_raw.csv')
print('Feature matrix:', X.shape)
print('Target balance:', y.value_counts().to_dict())
print('Cleaning:', report.notes)
pre = build_preprocessor()
Xt = pre.fit_transform(X)
print('After preprocessing (one-hot expands categoricals):', Xt.shape)

Feature matrix: (302, 13)
Target balance: {1: 164, 0: 138}
Cleaning: ['Dropped 1 duplicate row(s).', 'Coerced ca==4 -> NaN: 5', 'Coerced thal==0 -> NaN: 2']
After preprocessing (one-hot expands categoricals): (302, 25)


## 2. Model development & tuning

Two classifiers, each tuned with **5-fold stratified** `GridSearchCV`, scored on
**ROC-AUC** (threshold-independent, good for mildly imbalanced data):

- **Logistic Regression** — strong linear baseline, interpretable coefficients.
  Grid: `C ∈ {0.05, 0.1, 0.5, 1, 5}`, `class_weight ∈ {None, balanced}`.
- **Random Forest** — captures non-linear interactions.
  Grid over `n_estimators, max_depth, min_samples_leaf, class_weight`.

Run the full pipeline from the repo root:

```bash
python src/train.py
```

This logs both runs to MLflow (SQLite backend), saves plots, and persists the best
pipeline to `models/heart_pipeline.joblib`.


## 3. Results (test set, 20% stratified hold-out)

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| **Logistic Regression** | **0.820** | 0.824 | 0.848 | 0.836 | **0.894** |
| Random Forest | 0.770 | 0.744 | **0.879** | 0.806 | 0.893 |

**Winner: Logistic Regression** — best ROC-AUC and accuracy, and its tuning chose
`class_weight='balanced'`, directly addressing the mild class imbalance. Notably the
Random Forest achieves the highest *recall* (0.879); in a clinical screening context
where missing a true case is costly, RF (or a lower decision threshold on LogReg)
would be a defensible alternative — a trade-off worth discussing in the report.


In [3]:
# Load the winning model's metadata produced by src/train.py
meta = json.load(open('../models/model_metadata.json'))
print(json.dumps(meta, indent=2))

{
  "winner": "logistic_regression",
  "metrics": {
    "accuracy": 0.8197,
    "precision": 0.8235,
    "recall": 0.8485,
    "f1": 0.8358,
    "roc_auc": 0.8939
  },
  "best_params": {
    "C": 0.5,
    "class_weight": "balanced"
  },
  "feature_order": [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak",
    "cp",
    "restecg",
    "slope",
    "thal",
    "ca",
    "sex",
    "fbs",
    "exang"
  ],
  "clean_notes": [
    "Dropped 1 duplicate row(s).",
    "Coerced ca==4 -> NaN: 5",
    "Coerced thal==0 -> NaN: 2"
  ],
  "n_train": 241,
  "n_test": 61
}


## 4. Experiment tracking (MLflow)

Every run logs: hyper-parameters, CV ROC-AUC, all four test metrics, the ROC curve,
the confusion matrix, a classification report, and the serialized model. Inspect with:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
# open http://127.0.0.1:5000
```

**Screenshots for the report:** capture the MLflow run comparison view and one run's
metrics/artifacts page.


## 5. Reproducibility (Task 4)

- Full pipeline (preprocess + classifier) saved as `models/heart_pipeline.joblib`.
- `models/model_metadata.json` records winner, metrics, params, feature order, and cleaning notes.
- `requirements.txt` pins all versions.
- Loading + predicting is verified in the training run and reused unchanged by the API (Phase 3).
